# Optimizing an ML Pipeline in Azure

Two approaches on the **Bank Marketing** dataset, then a comparison:
1. **Scikit-learn Logistic Regression** with hyperparameters tuned by **HyperDrive**.
2. **AutoML** on the same cleaned data.


## 1. Connect to the workspace and create an experiment

In [1]:
from azureml.core import Workspace, Experiment

ws = Workspace.from_config()
exp = Experiment(workspace=ws, name='udacity-project')

print('Workspace name : ' + ws.name,
      'Azure region   : ' + ws.location,
      'Subscription id : ' + ws.subscription_id,
      'Resource group  : ' + ws.resource_group, sep='\n')

run = exp.start_logging()

Workspace name : quick-starts-ws-300810
Azure region   : westeurope
Subscription id : 610d6e37-4747-4a20-80eb-3aad70a55f43
Resource group  : aml-quickstarts-300810


## 2. Create (or attach to) a compute cluster

In [2]:
from azureml.core.compute import ComputeTarget, AmlCompute
from azureml.core.compute_target import ComputeTargetException

cluster_name = 'cpu-cluster'

try:
    cpu_cluster = ComputeTarget(workspace=ws, name=cluster_name)
    print('Found existing cluster, using it.')
except ComputeTargetException:
    compute_config = AmlCompute.provisioning_configuration(
        vm_size='STANDARD_D2_V2', max_nodes=4)
    cpu_cluster = ComputeTarget.create(ws, cluster_name, compute_config)

cpu_cluster.wait_for_completion(show_output=True)

InProgress...
SucceededProvisioning operation finished, operation "Succeeded"
Succeeded
AmlCompute wait for completion finished

Minimum number of nodes requested have been provisioned


## 3. HyperDrive configuration

In [4]:
from azureml.core import ScriptRunConfig, Environment
from azureml.train.hyperdrive.run import PrimaryMetricGoal
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.runconfig import HyperDriveConfig
from azureml.train.hyperdrive.parameter_expressions import uniform, choice
import os

# Parameter sampler: random search over C and max_iter.
ps = RandomParameterSampling({
    '--C': uniform(0.01, 10.0),
    '--max_iter': choice(50, 100, 150, 200)
})

# Early-termination policy: aggressively stop poorly performing runs.
policy = BanditPolicy(evaluation_interval=2, slack_factor=0.1)

# Environment for train.py (uses conda_dependencies.yml in this folder).
sklearn_env = Environment.from_conda_specification(
    name='sklearn-env', file_path='conda_dependencies.yml')

src = ScriptRunConfig(source_directory='.',
                      script='train.py',
                      compute_target=cpu_cluster,
                      environment=sklearn_env)

hyperdrive_config = HyperDriveConfig(
    run_config=src,
    hyperparameter_sampling=ps,
    policy=policy,
    primary_metric_name='Accuracy',
    primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
    max_total_runs=20,
    max_concurrent_runs=4)

## 4. Submit the HyperDrive run

In [7]:
hdr = exp.submit(config=hyperdrive_config)
try:
    from azureml.widgets import RunDetails
    RunDetails(hdr).show()
except Exception:
    pass  # widget unavailable on this kernel; track progress in Studio > Jobs
hdr.wait_for_completion(show_output=True)


RunId: HD_bc0e3e06-8e84-455d-af4c-1009ee831ec9
Web View: https://ml.azure.com/runs/HD_bc0e3e06-8e84-455d-af4c-1009ee831ec9?wsid=/subscriptions/610d6e37-4747-4a20-80eb-3aad70a55f43/resourcegroups/aml-quickstarts-300810/workspaces/quick-starts-ws-300810&tid=660b3398-b80e-49d2-bc5b-ac1dc93b5254

Streaming azureml-logs/hyperdrive.txt

[2026-06-26T06:44:53.8726209Z][GENERATOR][DEBUG]Sampled 4 jobs from search space 
[2026-06-26T06:44:55.4873647Z][SCHEDULER][INFO]Scheduling job, id='HD_bc0e3e06-8e84-455d-af4c-1009ee831ec9_0' 
[2026-06-26T06:44:55.6502620Z][SCHEDULER][INFO]Scheduling job, id='HD_bc0e3e06-8e84-455d-af4c-1009ee831ec9_2' 
[2026-06-26T06:44:55.6510917Z][SCHEDULER][INFO]Scheduling job, id='HD_bc0e3e06-8e84-455d-af4c-1009ee831ec9_3' 
[2026-06-26T06:44:55.6516868Z][SCHEDULER][INFO]Scheduling job, id='HD_bc0e3e06-8e84-455d-af4c-1009ee831ec9_1' 
[2026-06-26T06:44:56.5075194Z][SCHEDULER][INFO]Successfully scheduled a job. Id='HD_bc0e3e06-8e84-455d-af4c-1009ee831ec9_0' 
[2026-06-26T06:4

{'runId': 'HD_bc0e3e06-8e84-455d-af4c-1009ee831ec9',
 'target': 'cpu-cluster',
 'status': 'Completed',
 'startTimeUtc': '2026-06-26T06:44:52.274251Z',
 'endTimeUtc': '2026-06-26T07:10:14.33016Z',
 'services': {},
 'properties': {'primary_metric_config': '{"name":"Accuracy","goal":"maximize"}',
  'resume_from': 'null',
  'runTemplate': 'HyperDrive',
  'azureml.runsource': 'hyperdrive',
  'platform': 'AML',
  'ContentSnapshotId': '04f642cf-1408-4c40-88ef-a92be24db7ba',
  'user_agent': 'python/3.10.11 (Linux-6.8.0-1044-azure-x86_64-with-glibc2.35) msrest/0.7.1 Hyperdrive.Service/1.0.0 Hyperdrive.SDK/core.1.61.0',
  'best_child_run_id': 'HD_bc0e3e06-8e84-455d-af4c-1009ee831ec9_11',
  'score': '0.9088012139605464',
  'best_metric_status': 'Succeeded',
  'best_data_container_id': 'dcid.HD_bc0e3e06-8e84-455d-af4c-1009ee831ec9_11'},
 'inputDatasets': [],
 'outputDatasets': [],
 'runDefinition': {'configuration': None,
  'attribution': None,
  'telemetryValues': {'amlClientType': 'azureml-sdk-t

## 5. Register the best HyperDrive model

In [8]:
best_run = hdr.get_best_run_by_primary_metric()
best_run_metrics = best_run.get_metrics()

print('Best run id :', best_run.id)
print('Best run metrics :', best_run_metrics)
print('Best Accuracy :', best_run_metrics['Accuracy'])

best_run.register_model(
    model_name='hyperdrive_best_model',
    model_path='outputs/model.joblib')

Best run id : HD_bc0e3e06-8e84-455d-af4c-1009ee831ec9_11
Best run metrics : {'Accuracy': 0.9088012139605463, 'Regularization Strength:': 5.1009432630608, 'Max iterations:': 150}
Best Accuracy : 0.9088012139605463


Model(workspace=Workspace.create(name='quick-starts-ws-300810', subscription_id='610d6e37-4747-4a20-80eb-3aad70a55f43', resource_group='aml-quickstarts-300810'), name=hyperdrive_best_model, id=hyperdrive_best_model:1, version=1, tags={}, properties={})

## 6. AutoML — prepare the data

Reuse `clean_data` from `train.py` so AutoML trains on exactly the same features.

In [9]:
# Reuse the exact same loader + cleaning as train.py so AutoML
# trains on identical features. Falls back to pandas if needed.
from train import clean_data, ds, DATA_URL
import pandas as pd

data_src = ds if ds is not None else pd.read_csv(DATA_URL)
x, y = clean_data(data_src)
data = pd.concat([x, y.rename('y')], axis=1)  # AutoML needs the label inside the frame


{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


## 7. AutoML configuration and run

In [14]:
from azureml.train.automl import AutoMLConfig
from azureml.core import Dataset
import os

# Sanitize column names (AutoML/parquet dislike '.' in names)
data.columns = [c.replace('.', '_') for c in data.columns]

# Save + register a TabularDataset (rubric-friendly, also fine for local run)
os.makedirs('automl_data', exist_ok=True)
data.to_csv('automl_data/bankmarketing_clean.csv', index=False)
datastore = ws.get_default_datastore()
datastore.upload(src_dir='automl_data', target_path='bankmarketing_clean',
                overwrite=True, show_progress=True)
training_dataset = Dataset.Tabular.from_delimited_files(
    path=[(datastore, 'bankmarketing_clean/bankmarketing_clean.csv')])
training_dataset = training_dataset.register(
    workspace=ws, name='bankmarketing_clean', create_new_version=True)

# Run AutoML LOCALLY (no compute_target) to avoid the broken remote
# curated-environment image build (urllib3 conflict in AutoML-Non-Prod:1.61.0).
automl_config = AutoMLConfig( 
    experiment_timeout_minutes=30,
    task='classification',
    primary_metric='accuracy',
    training_data=training_dataset,
    label_column_name='y',
    n_cross_validations=5,
    _ignore_package_version_incompatibilities=True)


automl_run = exp.submit(automl_config, show_output=True)
automl_run.wait_for_completion(show_output=True)


Uploading an estimated of 1 files
Uploading automl_data/bankmarketing_clean.csv
Uploaded automl_data/bankmarketing_clean.csv, 1 files out of an estimated total of 1
Uploaded 1 files
No run_configuration provided, running on local with default configuration
Running in the active local environment.


2026-06-26 08:10:45.496901: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-26 08:10:45.497038: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-26 08:10:46.816005: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.


Experiment,Id,Type,Status,Details Page,Docs Page
udacity-project,AutoML_b34dda0c-d53d-4cb0-a62e-248eeb82ff2d,automl,Preparing,Link to Azure Machine Learning studio,Link to Documentation


Current status: DatasetEvaluation. Gathering dataset statistics.
Current status: FeaturesGeneration. Generating features for the dataset.
Current status: DatasetFeaturization. Beginning to fit featurizers and featurize the dataset.
Current status: DatasetFeaturizationCompleted. Completed fit featurizers and featurizing the dataset.
Current status: DatasetBalancing. Performing class balancing sweeping
Current status: DatasetCrossValidationSplit. Generating individually featurized CV splits.

********************************************************************************************
DATA GUARDRAILS: 

TYPE:         Class balancing detection
STATUS:       ALERTED
DESCRIPTION:  To decrease model bias, please cancel the current run and fix balancing problem.
              Learn more about imbalanced data: https://aka.ms/AutomatedMLImbalancedData
DETAILS:      Imbalanced data can lead to a falsely perceived positive effect of a model's accuracy because the input data has bias towards one cl

2026/06/26 08:12:34 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026-06-26:08:42:30,486 WARNING  [_docstring_wrapper.py:27] Class StackEnsembleClassifier: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
2026-06-26:08:42:31,189 WARNING  [_docstring_wrapper.py:27] Class StackEnsembleClassifier: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
2026-06-26:08:42:31,190 WARNING  [_docstring_wrapper.py:27] Class StackEnsembleClassifier: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
2026-06-26:08:42:31,884 WARNING  [_docstring_wrapper.py:27] Class StackEnsembleClassifier: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
2026-06-26:

Experiment,Id,Type,Status,Details Page,Docs Page
udacity-project,AutoML_b34dda0c-d53d-4cb0-a62e-248eeb82ff2d,automl,Completed,Link to Azure Machine Learning studio,Link to Documentation




********************************************************************************************
DATA GUARDRAILS: 

TYPE:         Class balancing detection
STATUS:       ALERTED
DESCRIPTION:  To decrease model bias, please cancel the current run and fix balancing problem.
              Learn more about imbalanced data: https://aka.ms/AutomatedMLImbalancedData
DETAILS:      Imbalanced data can lead to a falsely perceived positive effect of a model's accuracy because the input data has bias towards one class.
+------------------------------+--------------------------------+--------------------------------------+
|Size of the smallest class    |Name/Label of the smallest class|Number of samples in the training data|
+==============================+================================+======================================+
|3692                          |True                            |32950                                 |
+------------------------------+--------------------------------+----

{'runId': 'AutoML_b34dda0c-d53d-4cb0-a62e-248eeb82ff2d',
 'target': 'local',
 'status': 'Completed',
 'startTimeUtc': '2026-06-26T08:11:22.422099Z',
 'endTimeUtc': '2026-06-26T08:42:49.417064Z',
 'services': {},
 'warnings': [{'source': 'JasmineService',
   'message': 'Experiment timeout reached, hence experiment stopped. Current experiment timeout: 0 hour(s) 30 minute(s)'}],
 'properties': {'num_iterations': '1000',
  'training_type': 'TrainFull',
  'acquisition_function': 'EI',
  'primary_metric': 'accuracy',
  'train_split': '0',
  'acquisition_parameter': '0',
  'num_cross_validation': '5',
  'target': 'local',
  'AMLSettingsJsonString': '{"path":null,"name":"udacity-project","subscription_id":"610d6e37-4747-4a20-80eb-3aad70a55f43","resource_group":"aml-quickstarts-300810","workspace_name":"quick-starts-ws-300810","region":"westeurope","compute_target":"local","spark_service":null,"azure_service":"Microsoft.AzureNotebookVM","many_models":false,"pipeline_fetch_max_batch_size":1,"ena

## 8. Retrieve and register the best AutoML model

In [17]:
best_automl_run, fitted_model = automl_run.get_output()
print(best_automl_run)
print(fitted_model)

best_automl_run.register_model(
    model_name='automl_best_model',
    model_path='outputs/')

Run(Experiment: udacity-project,
Id: AutoML_7f2f9b9b-16cd-4c94-b862-f9ac5fb901d8_11,
Type: None,
Status: Completed)
Pipeline(steps=[('datatransformer',
                 DataTransformer(enable_dnn=False, enable_feature_sweeping=True, is_cross_validation=True, working_dir='/mnt/batch/tasks/shared/LS_root/mounts/clusters/excercise/code/Users/odl_user_300810/nd00333_AZMLND_Optimizing_a_Pipeline')),
                ('StandardScalerWrapper',
                 StandardScalerWrapper(copy=True, with_mean=False, with_std=False)),
                ('XGBoostClassifier',
                 XGBoostClassifier(booster='gbtree', colsample_bytree=0.6, eta=0.3, gamma=0, max_depth=6, max_leaves=0, n_estimators=10, objective='reg:logistic', problem_info=ProblemInfo(gpu_training_param_dict={'processing_unit_type': 'cpu'}), reg_alpha=0.3125, reg_lambda=2.3958333333333335, subsample=1, tree_method='auto'))])
Y_transformer(['LabelEncoder', LabelEncoder()])


Model(workspace=Workspace.create(name='quick-starts-ws-300810', subscription_id='610d6e37-4747-4a20-80eb-3aad70a55f43', resource_group='aml-quickstarts-300810'), name=automl_best_model, id=automl_best_model:2, version=2, tags={}, properties={})

## 9. Clean up the compute cluster

In [18]:
cpu_cluster.delete()